### Installation and Setup ###

In [ ]:
# Setup tools to avoid package conflicts

!pip install -U pip setuptools
!bash install_packages.sh

In [ ]:
import torch
import transformers
import sentencepiece
import numpy as np
import google.protobuf  # Should import without error
print(google.protobuf.__version__)  # Should print 4.25.3
print(torch.cuda.is_available())  # True
print(torch.cuda.device_count())  # 2
print(np.__version__)  # Should print 1.26.4
print(torch.__version__)  # Should print 2.2.1

In [ ]:
# Hugging face authentication
from huggingface_hub import login
# Replace with your actual token from huggingface.co/settings/tokens
login(token="YOUR_HF_TOKEN")

### Display dataset and token count ###

In [ ]:
# Analyze dataset examples length
from datasets import load_dataset

MAX_SEQ_LENGTH = 8192

def estimate_tokens(text):
    """Rough estimation of tokens (approx 4 characters per token)"""
    if not text:
        return 0
    return len(text) // 4

train_dataset = load_dataset("json", data_files="train_codeLlama.jsonl", split="train") 

def format_completion_example(example):
    prompt = example.get("prompt", "")
    completion = example.get("completion", "")
    return {"text": f"{prompt}{completion}"}
    
train_dataset = train_dataset.map(format_completion_example, batched=False)
token_counts = [estimate_tokens(example['text']) for example in train_dataset]

# Plot token length distribution
plt.figure(figsize=(10, 5))
plt.hist(token_counts, bins=40, color='skyblue', edgecolor='black')
plt.axvline(x=MAX_SEQ_LENGTH, color='red', linestyle='--', label=f'MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH}')
plt.title('🧮 Token Length Distribution in Training Dataset')
plt.xlabel('Estimated Token Count (approx)')
plt.ylabel('Number of Examples')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Stats
avg_length = np.mean(token_counts)
over_limit = sum(c > MAX_SEQ_LENGTH for c in token_counts)

print(f"📏 Average example length: {avg_length:.2f} tokens")
print(f"⚠️  Examples longer than {MAX_SEQ_LENGTH} tokens: {over_limit} out of {len(train_dataset)}")

### Training using normal trainer ###

In [ ]:
# Complete Training Script 1 using normal trainer

# Step 1: Necassary Imports
import torch
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset

# Configure logging
import logging
import os
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
)
logger = logging.getLogger(__name__)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def print_trainable_parameters(model):
    """Prints number and percentage of trainable parameters in the model."""
    trainable_params = 0
    all_params = 0
    
    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    percentage = 100 * trainable_params / all_params
    logger.info(f"Trainable params: {trainable_params:,} / {all_params:,} "
                f"({percentage:.2f}% of total)")
    return trainable_params, all_params


# Step 2: Verify CUDA and Set base Model
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}, GPUs: {torch.cuda.device_count()}")
BASE_MODEL = "codellama/CodeLlama-7b-hf" # base model codellama

# Step 3: Load dataset
dataset = load_dataset("json", data_files="train_codeLlama.jsonl", split="train") 

# Step 4: Preprocess dataset and split
def format_and_tokenize(example):
    prompt = example.get("prompt", "")
    completion = example.get("completion", "")
    text = f"{prompt}{completion}"
    
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=8192,  # or adjust based on your model (e.g. 4096 or 8192)
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized
    
dataset = dataset.map(format_and_tokenize, batched=False, remove_columns=dataset.column_names)

split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# Display raw sample
#print("\n🔎 Sample from raw dataset:")
#print(dataset[0])

#print("\n🔎 Sample from train and eval datasets:")
#print(train_dataset[0])
#print(eval_dataset[0])

# Step 5: Set 4-bit quantization config (QLORA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16  # BF16 compute for H100
)

# Step 6: Load model
logger.info(f"🧠 Loading model {BASE_MODEL} with 4-bit quantization config")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    #torch_dtype=torch.bfloat16  # Required for H100, aligns with bnb_config
)

# Step 7: Load tokenizer
logger.info(f"📦 Loading tokenizer for model: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Step 8: Prepare model for 4-bit training
model = prepare_model_for_kbit_training(model)

# Step 9: Set LoRA config and get PEFT model for training
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()
# Enable input gradients
model.enable_input_require_grads()

# ✅ Log trainable params
print_trainable_parameters(model)
logger.info("✅ Model ready for LoRA training with 4-bit quantization on H100")

# Step 10: Set training arguments
training_args = TrainingArguments(
    output_dir = f"./results/codellama_normalTrain",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2,
    num_train_epochs = 3,
    learning_rate = 2e-4,
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    warmup_ratio = 0.03,
    evaluation_strategy = "steps",
    save_strategy = "steps",
    logging_steps = 10,
    eval_steps = 100,
    save_steps = 250,
    bf16 = True, 
    logging_dir = os.path.join(f"./results/codellama_normalTrain", "logs"),
    report_to="none",  # or "wandb" if using Weights & Biases
    save_total_limit = 2,
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# Step 11: Set up trainer
trainer = Trainer(
    model=model,
    train_dataset = dataset,
    tokenizer = tokenizer,
    args=training_args,
    # max_seq_length = 8192
)

# Step 12: Train the model
trainer.train()

# Step 13: Save the model and tokenizer
model.save_pretrained(os.path.join(f"./results/codellama_normalTrain", "final_model"))
tokenizer.save_pretrained(os.path.join(f"./results/codellama_normalTrain", "final_model"))

### Training using SFT Trainer ###

In [ ]:
# Complete Training Script 2 using SFT trainer

# Step 1: Necassary Imports
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from trl import SFTTrainer
from datasets import load_dataset

# Configure logging
import logging
import os
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
)
logger = logging.getLogger(__name__)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Prints trainable parameters of the model
def print_trainable_parameters(model):
    """Prints number and percentage of trainable parameters in the model."""
    trainable_params = 0
    all_params = 0
    
    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    percentage = 100 * trainable_params / all_params
    logger.info(f"Trainable params: {trainable_params:,} / {all_params:,} "
                f"({percentage:.2f}% of total)")
    return trainable_params, all_params

# Step 2: Verify CUDA and Set base Model
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}, GPUs: {torch.cuda.device_count()}")
BASE_MODEL = "codellama/CodeLlama-7b-hf" # base model codellama

# Step 3: Load dataset
dataset = load_dataset("json", data_files="train_codeLlama.jsonl", split="train") 

# Step 4: Preprocess dataset and split
def format_completion_example(example):
    prompt = example.get("prompt", "")
    completion = example.get("completion", "")
    return {"text": f"{prompt}{completion}"}
    
dataset = dataset.map(format_completion_example, batched=False)

split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# Display raw sample
#print("\n🔎 Sample from raw dataset:")
#print(dataset[0])

#print("\n🔎 Sample from train and eval datasets:")
#print(train_dataset[0])
#print(eval_dataset[0])

# Step 5: Set 4-bit quantization config (QLORA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16  # BF16 compute for H100
)

# Step 6: Load model
logger.info(f"🧠 Loading model {BASE_MODEL} with 4-bit quantization config")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    #torch_dtype=torch.bfloat16  # Required for H100, aligns with bnb_config
)

# Step 7: Load tokenizer
logger.info(f"📦 Loading tokenizer for model: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Step 8: Prepare model for 4-bit training
model = prepare_model_for_kbit_training(model)

# Step 9: Set LoRA config and get PEFT model for training
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()
# Enable input gradients
model.enable_input_require_grads()

# ✅ Log trainable params
print_trainable_parameters(model)
logger.info("✅ Model ready for LoRA training with 4-bit quantization on H100")

# Step 10: Set training arguments
training_args = TrainingArguments(
    output_dir = f"./results/codellama_SFTTrain",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2,
    num_train_epochs = 3,
    learning_rate = 2e-4,
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    warmup_ratio = 0.03,
    evaluation_strategy = "steps",
    save_strategy = "steps",
    logging_steps = 10,
    eval_steps = 100,
    save_steps = 250,
    bf16 = True, 
    logging_dir = os.path.join(f"./results/codellama_SFTTrain", "logs"),
    report_to="none",  # or "wandb" if using Weights & Biases
    save_total_limit = 2,
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# Step 11: Set up trainer
trainer = SFTTrainer(
    model=model,
    train_dataset = dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    tokenizer = tokenizer,
    args=training_args,
    max_seq_length = 8192
)

# Step 12: Train the model
trainer.train()

# Step 13: Save the model and tokenizer
model.save_pretrained(os.path.join(f"./results/codellama_SFTTrain", "final_model"))
tokenizer.save_pretrained(os.path.join(f"./results/codellama_SFTTrain", "final_model"))

### Batch Inference Execution ###

In [ ]:
#!/usr/bin/env python3
"""
Generate and Compare Responses from Base and Fine-Tuned DeepSeek Models

This script:
1. Loads prompts from a JSON file
2. Generates code responses using both base and fine-tuned DeepSeek Coder models
3. Saves results to:
   - Generates Individual C files (one per prompt)
   - A consolidated JSON file with all results

Created: 2025-05-18 01:20:17
Author: Ali Hassan
"""

import json
import os
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, AutoPeftModelForCausalLM
import re
from datetime import datetime
import getpass
from tqdm import tqdm

import gc
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Configuration
INPUT_JSON = "Test_Train_Samples_Evaluation_Set2_17May2025.json"
OUTPUT_JSON = "CodeLlama_responses_eval2_17May2025.json"
OUTPUT_DIR = "codellama_generated_code"
BASE_MODEL_ID = "codellama/CodeLlama-7b-hf"
FINETUNED_MODEL_PATH = "./model"  # Adjust based on your path
RANDOM_SEED = 42
TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
USERNAME = "Ali Hassan"
EXECUTION_DEVICE = "GPU" if torch.cuda.is_available() else "CPU"

def setup_directory(directory):
    """Create output directory if it doesn't exist."""
    os.makedirs(directory, exist_ok=True)
    print(f"Output directory set up: {directory}")

def load_json(file_path):
    """Load examples from a JSON file containing a list of objects."""
    print(f"Loading data from {file_path}...")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Input file not found: {file_path}")

    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            examples = json.load(f)
            if not isinstance(examples, list):
                raise ValueError("Expected a list of JSON objects in the file.")
        except json.JSONDecodeError as e:
            raise ValueError(f"Error parsing JSON: {e}")

    print(f"Successfully loaded {len(examples)} examples.")
    return examples

def load_base_model():
    """Load the base CodeLlama model with 4-bit quantization."""
    print("\n1. Loading base CodeLlama model...")
    start_time = time.time()
    
    # Create 4-bit quantization config (same as fine-tuning)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    # Load the base model with 4-bit quantization
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    if base_tokenizer.pad_token is None:
        base_tokenizer.pad_token = base_tokenizer.eos_token
    
    print(f"   Base model loaded in {time.time() - start_time:.2f} seconds")
    
    return base_model, base_tokenizer

def load_finetuned_model():
    """Load the fine-tuned CodeLlama model with 4-bit quantization."""
    print("\n2. Loading fine-tuned CodeLlama model...")
    start_time = time.time()
    
    # Create 4-bit quantization config (same as fine-tuning)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    # Load the base model with 4-bit quantization first
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    # Then load the PEFT adapter
    finetuned_model = PeftModel.from_pretrained(
        base_model,
        FINETUNED_MODEL_PATH
    )
    
    finetuned_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    if finetuned_tokenizer.pad_token is None:
        finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token
    
    print(f"   Fine-tuned model loaded in {time.time() - start_time:.2f} seconds")
    
    return finetuned_model, finetuned_tokenizer

def format_prompt_for_codellama(prompt):
    """Format a prompt for CodeLlama code generation."""
    # CodeLlama works well with a simple instruction format
    formatted = f"<s>[INST] You are a Raspberry Pi C Code Expert. Write a C code for Raspberry Pi that accomplishes the following task:\n\n{prompt} [/INST]\n\n"
    return formatted

def generate_codellama_response(model, tokenizer, prompt, cal_max_new_tokens = 2048):
    """Generate a response from a CodeLlama model."""
    formatted_prompt = format_prompt_for_codellama(prompt)
    
    # Tokenize the prompt
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    input_length = len(inputs["input_ids"][0])
    
    print(f"   Input length: {input_length} tokens")
    
    with torch.no_grad():
        start_time = time.time()
        
        outputs = model.generate(
            **inputs,
            max_new_tokens=cal_max_new_tokens,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
        
        generation_time = time.time() - start_time
    
    # Decode the full response
    full_response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    
    # Clean up the response
    clean_response = extract_clean_code(full_response)
    
    print(f"   Generation time: {generation_time:.2f} seconds")
    return full_response, clean_response, generation_time

def extract_clean_code(response):
    """Extract clean C code from a model response, removing explanatory text."""
    
    # --- Extract C code block ---
    code_match = re.search(r"```c\n(.*?)```", response, re.DOTALL)
    if not code_match:
        code_match = re.search(r"```(.*?)```", response, re.DOTALL)
    if code_match:
        return code_match.group(1).strip()

    # --- Fallback: Extract code based on C syntax patterns ---
    lines = response.split('\n')
    code_lines = []
    in_code = False

    explanation_patterns = [
        r'^(Here\'s|This|Let\'s|Now|First|The code|I\'ll|To|We|That\'s|This will)',
        r'.*\b(explains|provides|shows|demonstrates|illustrates)\b.*',
        r'^In this code',
        r'^This program',
    ]

    for line in lines:
        stripped = line.strip()

        # Skip empty or explanatory lines unless already in code
        if not stripped and not in_code:
            continue
        if any(re.match(pattern, stripped, re.IGNORECASE) for pattern in explanation_patterns) and not in_code:
            continue

        # Detect C code patterns
        if re.match(r'^(#include|int\s+main|void\s+main|void\s+\w+|int\s+\w+|float\s+\w+|char\s+\w+|printf|scanf|for\s*\(|while\s*\(|if\s*\(|return\b)', stripped):
            in_code = True
            code_lines.append(line)
        elif in_code or (stripped and not stripped.endswith('.') and not stripped.endswith('?')):
            code_lines.append(line)

    if code_lines:
        return '\n'.join(code_lines)

    return response

def save_to_file(example_id, prompt, source, base_response, finetuned_response, output_dir):
    """Save responses into three separate C files."""
    # File paths
    combined_file = os.path.join(output_dir, f"example_combined_id_{example_id}_{source}.c")
    base_file = os.path.join(output_dir, f"example_baseDeepSeek_id_{example_id}_{source}.c")
    finetuned_file = os.path.join(output_dir, f"example_finetuned_id_{example_id}_{source}.c")

    # --- Write combined file ---
    with open(combined_file, "w", encoding="utf-8") as f:
        f.write(f"""\
/*
 * Generated DeepSeek Coder responses for prompt ID: {example_id}
 * Source: {source}
 * Generated on: {TIMESTAMP}
 * Generated by: {USERNAME}
 */

/* Original prompt:
{prompt}
*/

/* -----------------------------------------
   Base DeepSeek Response ({BASE_MODEL_ID}):
   ----------------------------------------- */
{base_response}

/* -----------------------------------------
   Fine-Tuned DeepSeek Response:
   ----------------------------------------- */
{finetuned_response}
""")

    # --- Write base-only file ---
    with open(base_file, "w", encoding="utf-8") as f:
        f.write(f"""\
/*
 * Base DeepSeek Response for prompt ID: {example_id}
 * Source: {source}
 * Generated on: {TIMESTAMP}
 * Generated by: {USERNAME}
 */

/* Original prompt:
{prompt}
*/

{base_response}
""")

    # --- Write finetuned-only file ---
    with open(finetuned_file, "w", encoding="utf-8") as f:
        f.write(f"""\
/*
 * Fine-Tuned DeepSeek Response for prompt ID: {example_id}
 * Source: {source}
 * Generated on: {TIMESTAMP}
 * Generated by: {USERNAME}
 */

/* Original prompt:
{prompt}
*/

{finetuned_response}
""")

    return combined_file, base_file, finetuned_file

def main():
    """Main execution function with checkpointing."""
    print(f"=== CodeLlama Response Generation ===")
    print(f"Date/Time: {TIMESTAMP}")
    print(f"User: {USERNAME}")

    torch.manual_seed(RANDOM_SEED)
    setup_directory(OUTPUT_DIR)

    examples = load_json(INPUT_JSON)
    total_examples = len(examples)

    # Load checkpoint if exists
    completed_results = []
    if os.path.exists(OUTPUT_JSON):
        try:
            with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
                completed_results = json.load(f)
                print(f"🔁 Resuming from checkpoint. Already processed: {len(completed_results)} examples.")
        except Exception as e:
            print(f"⚠️ Error reading checkpoint: {e}")
            completed_results = []

    start_index = len(completed_results)
    base_model, base_tokenizer = load_base_model()
    finetuned_model, finetuned_tokenizer = load_finetuned_model()

    start_time = time.time()

    for i, example in enumerate(tqdm(examples[start_index:], desc="🧠 Generating", unit="example", ncols=100), start=start_index):
        example_id = example["id"]
        prompt = example["input"]
        source = example["data_source"]
        code_reference = example["output"]

        output_len = len(code_reference)
        estimated_tokens = int((output_len / 4) * 2) if output_len > 0 else 2048
        cal_max_new_tokens = min(8192, max(128, estimated_tokens))

        tqdm.write(f"\n[{i+1}/{total_examples}] Processing prompt {example_id} (Source: {source}) (Allowed Tokens: {cal_max_new_tokens})")

        tqdm.write(f"   Generating base CodeLlama response...")
        base_response, base_code, base_time = generate_codellama_response(base_model, base_tokenizer, prompt, cal_max_new_tokens)

        tqdm.write(f"   Generating fine-tuned CodeLlama response...")
        finetuned_response, finetuned_code, finetuned_time = generate_codellama_response(finetuned_model, finetuned_tokenizer, prompt, cal_max_new_tokens)

        save_to_file(example_id, prompt, source, base_code, finetuned_code, OUTPUT_DIR)
        tqdm.write(f"   Saved to file set")

        example["base_response"] = base_response
        example["base_code"] = base_code
        example["base_execution_time"] = base_time
        example["finetuned_response"] = finetuned_response
        example["finetuned_code"] = finetuned_code
        example["finetuned_execution_time"] = finetuned_time
        example["allowed_max_new_tokens"] = cal_max_new_tokens

        completed_results.append(example)

        # Save checkpoint
        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump(completed_results, f, indent=2)

        # Clear memory to avoid memory buildup
        clear_memory()

        # ETA and timing feedback
        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1)
        remaining = avg_time * (total_examples - i - 1)
        eta_min, eta_sec = divmod(int(remaining), 60)
        tqdm.write(f"⏱️ Time/iter: {avg_time:.2f}s | ETA: {eta_min}m {eta_sec}s")

    print("\n=== CodeLlama Generation Complete ===")
    print(f"- Processed {total_examples} prompts")
    print(f"- C files saved to: {OUTPUT_DIR}/")
    print(f"- Results saved to: {OUTPUT_JSON}")
    print(f"- Timestamp: {TIMESTAMP}")
    print(f"- Generated by: {USERNAME}")
   
if __name__ == "__main__":
    main()